## Step 2.3:Asynchronous I/O & The Event Loop

## 1. The Event Loop & Cooperative Multitasking
**First Principle:** Multi-threading relies on *preemptive* multitasking, meaning the Operating System forcefully interrupts threads to share CPU time[cite: 4]. However, network I/O operations spend $>99\%$ of their lifecycle waiting for external packets across sockets, leaving CPU threads completely idle[cite: 4]. **Cooperative multitasking** solves this using a single thread running an **Event Loop**[cite: 4]. It relies on OS-level I/O multiplexing primitives (like `epoll` on Linux or `kqueue` on macOS) to monitor thousands of connections simultaneously[cite: 3, 4]. Tasks voluntarily yield control (`await`) when pausing for I/O, completely avoiding the memory overhead of OS thread context switches[cite: 4, 5].

> **Analogy:** Imagine a master chef (Single Thread / Event Loop) in a restaurant kitchen[cite: 4]. The chef puts bread into a toaster (Initiates I/O)[cite: 4]. Instead of freezing and staring at the toaster for two minutes (Preemptive/Blocking Thread), the chef immediately turns around to chop vegetables (Non-blocking Cooperative Task)[cite: 4]. When the toaster bell rings (an `epoll` socket readiness notification), the chef returns to grab the toast[cite: 4].

```text
                         +-----------------------------------+
                         |         THE EVENT LOOP            |
                         |  (Single thread continuously      |
                         |   polling OS multiplexer)         |
                         +-----------------------------------+
                                   |               ^
                   Dispatches task |               | Wakes up on I/O ready
                                   v               |
                         +-----------------------------------+
                         |  Task 1: Sockets / DB Network I/O |
                         |  Hits `await` -> Yields to Loop   |
                         +-----------------------------------+
                                   |
                                   v
                         +-----------------------------------+
                         |  Task 2: In-memory CPU validation |
                         |  Executes while Task 1 waits!     |
                         +-----------------------------------+


## 2. Coroutines vs. Standard Functions
**First Principle:** A standard subroutine has a single entry point and a single exit point; when invoked, its stack frame is pushed onto the call stack, runs to completion, and is destroyed.
A **Coroutine** is a state machine: a function that can suspend its execution frame, yield control back to a scheduler, preserve its entire local variable state, and resume execution exactly where it stopped.

> **Analogy:** 
> * **Standard Function (`def`):** A movie DVD that plays straight from beginning to end without stopping.
> * **Coroutine (`async def`):** A video game with a "Save State" feature. You pause at a checkpoint, walk away, let someone else play another game, and return later to resume at that exact coordinate with all your inventory intact.

```text
STANDARD FUNCTION:
Caller ----( invokes )----> [ Pushes Stack Frame ] ---> [ Runs to Return ] ---> [ Destroys Frame ]
                                                                                       |
Caller <-----------------( Returns final value )---------------------------------------+

COROUTINE (`async def`):
Caller ----( invokes )----> [ Returns Suspended Coroutine Object ] (Does NOT run body yet!)
                                      |
                      ( Event Loop advances execution )
                                      v
                            [ Enters Frame: Step 1 ]
                                      |
                               ( Hits `await` )
                                      v
                   <-- [ Suspends Frame & Yields to Loop ] --> (Other code runs)
                                      |
                         ( I/O Event Resolves: Wake up )
                                      v
                            [ Resumes Frame: Step 2 ]
                                      |
Caller <----------------- [ Reaches Return / Finishes ]

In [4]:
import inspect

def regular_sync_function():
    return "Hello from Sync"

async def async_coroutine_function():
    return "Hello from Async"

# 1. Calling a sync function executes it immediately
sync_result = regular_sync_function()
print(f"Sync result: {sync_result} | Type: {type(sync_result)}")

# 2. Calling an async function ONLY creates the coroutine object
coro_object = async_coroutine_function()
print(f"Async result: {coro_object} | Type: {type(coro_object)}")

# 3. Inspect internal coroutine state[cite: 4]
print(f"Is it a coroutine? {inspect.iscoroutine(coro_object)}")
print(f"Coroutine state: {inspect.getcoroutinestate(coro_object)}")

# Close the un-awaited coroutine to suppress runtime warnings
coro_object.close()

Sync result: Hello from Sync | Type: <class 'str'>
Async result: <coroutine object async_coroutine_function at 0x0000017121B6BD70> | Type: <class 'coroutine'>
Is it a coroutine? True
Coroutine state: CORO_CREATED


## 3. The Core Hierarchy: Awaitable vs. Future vs. Task
**First Principle:** Python's asynchronous model is built on an interface hierarchy called **Awaitables** (any object implementing the `__await__()` API):

*   **Coroutine:** The raw state machine created by `async def`. It has no knowledge of when it will be scheduled; it must be driven manually or handed to the loop.
*   **`asyncio.Future`:** A low-level container representing an eventual result of an asynchronous operation that has not completed yet.
*   **`asyncio.Task`:** A subclass of Future that wraps a Coroutine and registers it directly on the Event Loop's run queue immediately upon creation.

> **Analogy:**
> * **Coroutine:** A recipe card for baking a cake.
> * **Future:** A vibrating buzzer given to you at a restaurant counter. It starts empty, but when the kitchen finishes your order, it lights up and holds your food ticket.
> * **Task:** Placing that recipe card directly onto the chef's active ticket rail. The chef starts cooking it in the background immediately, and you hold the buzzer (Future) to know when it finishes.

```text
                            +--------------------+
                            |     Awaitable      |
                            |  (__await__() API) |
                            +--------------------+
                                      |
                 +--------------------+--------------------+
                 |                                         |
                 v                                         v
       +--------------------+                    +--------------------+
       |     Coroutine      |                    |       Future       |
       | (async def state   |                    | (Low-level result  |
       |     machine)       |                    |    placeholder)    |
       +--------------------+                    +--------------------+
                                                           ^
                                                           | (Inherits)
                                                 +--------------------+
                                                 |        Task        |
                                                 | (Wraps Coroutine + |
                                                 |  auto-schedules on |
                                                 |    Event Loop)     |
                                                 +--------------------+

In [5]:
import asyncio

async def sample_coroutine():
    await asyncio.sleep(0.1)
    return "Payload Data"

async def demonstrate_hierarchy():
    # 1. Raw Future demo: Manually setting a result
    loop = asyncio.get_running_loop()
    future_obj = loop.create_future()
    
    print(f"Initial Future State: Done={future_obj.done()}")
    future_obj.set_result("Manual Future Resolved!") # Manually resolve the future
    print(f"Resolved Future State: Done={future_obj.done()} | Value={await future_obj}")
    
    # 2. Task demo: Wrapping a coroutine into an auto-scheduled Task
    task_obj = asyncio.create_task(sample_coroutine())
    print(f"Task created: {task_obj}")
    print(f"Is Task an instance of Future? {isinstance(task_obj, asyncio.Future)}")
    
    result = await task_obj
    print(f"Task completed with result: {result}")

await demonstrate_hierarchy()

Initial Future State: Done=False
Resolved Future State: Done=True | Value=Manual Future Resolved!
Task created: <Task pending name='Task-46' coro=<sample_coroutine() running at C:\Users\shubh\AppData\Local\Temp\ipykernel_11984\1796405723.py:3>>
Is Task an instance of Future? True
Task completed with result: Payload Data


## 4. Execution Control: `await` vs. `create_task` vs. `gather`
**First Principle:** How you handle a coroutine dictates whether execution runs **sequentially (in series)** or **concurrently (in parallel time-slicing)**:

*   **Direct `await`:** Sequential. The current execution frame pauses immediately and does not proceed to the next line until the coroutine finishes.
*   **`asyncio.create_task()`:** Fire-and-Schedule. Puts the coroutine onto the Event Loop's ready queue immediately, allowing the caller to continue executing subsequent lines.
*   **`asyncio.gather()`:** Concurrent Batching. Converts multiple awaitables into tasks, schedules them concurrently, and pauses until *all* of them have resolved in order.

```text
TIMELINE COMPARISON:

1. Direct `await` (Sequential Execution): Total Time = 2s + 2s = 4s
Thread: |-- [ Await Task A (2s) ] --|-- [ Await Task B (2s) ] --|

2. `asyncio.create_task` / `asyncio.gather` (Concurrent Execution): Total Time = max(2s, 2s) = 2s
Thread: |-- [ Start Task A (2s) ] ------------------------------|
        |-- [ Start Task B (2s) ] (Overlapped I/O) -------------|
        \-- [ Event Loop resumes whichever finishes first ] ----/

In [6]:
import asyncio
import time

async def fetch_remote_resource(service_name, delay):
    print(f"[{time.strftime('%X')}] -> Starting request to: {service_name}")
    await asyncio.sleep(delay)  # Non-blocking network I/O simulation
    print(f"[{time.strftime('%X')}] <- Received response from: {service_name}")
    return f"{service_name}_DATA"

async def benchmark_execution_patterns():
    print("=== PATTERN 1: Sequential Execution (Direct `await`) ===")
    t0 = time.time()
    res1 = await fetch_remote_resource("DB_Users", 1.0)
    res2 = await fetch_remote_resource("Auth_Service", 1.0)
    print(f"Sequential Execution Total Time: {time.time() - t0:.2f}s\n")

    print("=== PATTERN 2: Concurrent Execution (`asyncio.create_task`) ===")
    t0 = time.time()
    # Schedules both immediately in the background:
    task_a = asyncio.create_task(fetch_remote_resource("DB_Users", 1.0))
    task_b = asyncio.create_task(fetch_remote_resource("Auth_Service", 1.0))
    
    print(f"[{time.strftime('%X')}] Tasks are now running! Doing local CPU work...")
    await asyncio.sleep(0.2) 
    
    # Await their completion when results are needed
    res_a = await task_a
    res_b = await task_b
    print(f"Concurrent Tasks Total Time: {time.time() - t0:.2f}s\n")

    print("=== PATTERN 3: Concurrent Batching (`asyncio.gather`) ===")
    t0 = time.time()
    results = await asyncio.gather(
        fetch_remote_resource("DB_Users", 1.0),
        fetch_remote_resource("Auth_Service", 1.0),
        fetch_remote_resource("Payment_Gateway", 1.0)
    )
    print(f"Gather Results: {results}")
    print(f"Gather Total Time: {time.time() - t0:.2f}s")

await benchmark_execution_patterns()

=== PATTERN 1: Sequential Execution (Direct `await`) ===
[22:50:19] -> Starting request to: DB_Users
[22:50:20] <- Received response from: DB_Users
[22:50:20] -> Starting request to: Auth_Service
[22:50:21] <- Received response from: Auth_Service
Sequential Execution Total Time: 2.02s

=== PATTERN 2: Concurrent Execution (`asyncio.create_task`) ===
[22:50:21] Tasks are now running! Doing local CPU work...
[22:50:21] -> Starting request to: DB_Users
[22:50:21] -> Starting request to: Auth_Service
[22:50:22] <- Received response from: DB_Users
[22:50:22] <- Received response from: Auth_Service
Concurrent Tasks Total Time: 1.01s

=== PATTERN 3: Concurrent Batching (`asyncio.gather`) ===
[22:50:22] -> Starting request to: DB_Users
[22:50:22] -> Starting request to: Auth_Service
[22:50:22] -> Starting request to: Payment_Gateway
[22:50:23] <- Received response from: DB_Users
[22:50:23] <- Received response from: Auth_Service
[22:50:23] <- Received response from: Payment_Gateway
Gather Resul